# Artificial Intelligence Use and Time Savings
### An Analysis of Students and Professionals

The objective is to examine how daily hours of AI use relate to weekly time savings and compare those savings between students and professionals.

## Research Questions

1. How do daily hours of AI use relate to hours saved per week?
2. How does the distribution of weekly time savings differ between students and professionals?


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

path = '/content/drive/MyDrive/AI_Usage_and_Impact_on_Students_and_Professionals.csv'

df = pd.read_csv(path)

df.head()

In [ ]:
# Check dataset dimensions and columns
print('Rows and columns:', df.shape)

df.columns

## Dataset

**Source:** AI_Usage_and_Impact_on_Students_and_Professionals.csv, used for this activity. Its original source and data collection method are unknown.

**Format:** tabular data in a CSV file.

**Context:** AI use by students and professionals, including usage time and their experience.

**Observations:** 300 records. **Variables:** 19 columns.

**Unit of observation:** each row represents a record for a user identified by User_ID.

These results describe this dataset; they cannot be assumed to represent all students and professionals.


In [ ]:
# Select and rename variables
ndf = df[
    ['User_Type',
     'AI_Usage_Hours_Per_Day',
     'Time_Saved_Hours_Per_Week']
].copy()

ndf.rename(columns={
    'User_Type': 'user_type',
    'AI_Usage_Hours_Per_Day': 'usage',
    'Time_Saved_Hours_Per_Week': 'savings'
}, inplace=True)

ndf.head()

In [ ]:
# Clean hour values and convert to numbers
ndf['usage'] = pd.to_numeiqr(
    ndf['usage'].astype(str).str.replace('hrs', '', regex=False).str.strip(),
    errors='coerce'
)
ndf['savings'] = pd.to_numeiqr(
    ndf['savings'].astype(str).str.replace('hrs', '', regex=False).str.strip(),
    errors='coerce'
)
ndf['user_type'] = ndf['user_type'].str.strip().replace({
    'Student': 'Student',
    'Professional': 'Professional'
})

ndf.head()

In [ ]:
# Check missing values
ndf.isnull().sum()

In [ ]:
ndf['user_type'].value_counts()

## Variables of Interest

| Variable | Meaning | Type | Unit or Categories |
|---|---|---|---|
| user_type | Type of user | Binary nominal categorical | Student and Professional |
| usage | Daily AI usage time | Continuous numerical | Hours per day |
| savings | Weekly time saved | Continuous numerical | Hours per week |

Usage and savings are durations, not dates. This analysis is not a time series.


In [ ]:
# Overall statistics
ndf[['usage', 'savings']].describe().round(2)

In [ ]:
print('Mode of usage:')
print(ndf['usage'].mode())

print('Mode of savings:')
print(ndf['savings'].mode())

print('Variance:')
print(ndf[['usage', 'savings']].var().round(2))

print('Range:')
print(
    ndf[['usage', 'savings']].max()
    - ndf[['usage', 'savings']].min()
)

In [ ]:
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

## Study 1

What is the relationship between daily hours of AI use and hours saved per week?

I use a scatter plot. Each point represents a user with information available for both variables.


In [ ]:
# Scatter plot
pairs = ndf.dropna(subset=['usage', 'savings'])

plt.figure()

plt.scatter(
    pairs['usage'],
    pairs['savings'],
    color='steelblue',
    alpha=0.6,
    s=40
)

plt.title('Daily AI Use and Weekly Time Savings', fontsize=17)
plt.xlabel('AI Use (Hours per Day)', fontsize=13)
plt.ylabel('Time Saved (Hours per Week)', fontsize=13)

plt.tight_layout()
plt.savefig('usage_savings.png', dpi=200)
plt.show()

In [ ]:
# Correlation
print('Users analyzed:', len(pairs))

pairs[['usage', 'savings']].corr().round(3)

### Findings

1. The 283 complete records show a strong positive relationship: users who spend more hours using AI tend to report greater weekly time savings. The correlation is 0.779.
2. Values far from the majority, such as 23 hours of use per day, require review and may influence the correlation. The observed relationship does not establish cause and effect.


## Study 2

How are weekly time savings distributed among students and professionals?

I use a pivot table to compare statistics and a box plot to examine the median, spread, and potential outliers.


In [ ]:
# Pivot table
table = pd.pivot_table(
    ndf,
    index='user_type',
    values='savings',
    aggfunc=['count', 'mean', 'median', 'std']
)

table.round(2)

In [ ]:
plot_data = ndf.dropna(subset=['user_type', 'savings'])

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=plot_data,
    x='user_type',
    y='savings',
    color='lightblue'
)

plt.title('Weekly Time Savings by User Type', fontsize=17)
plt.xlabel('User Type', fontsize=13)
plt.ylabel('Time Saved (Hours per Week)', fontsize=13)

plt.tight_layout()
plt.savefig('savings_user_type.png', dpi=200)
plt.show()

### Findings

1. Students save an average of 6.10 hours per week, while professionals save 5.56. However, their medians are very similar: 5.00 and 5.10 hours, respectively.
2. Savings vary more among students. Their standard deviation is 4.38 hours, compared with 3.73 for professionals. The distributions overlap, so the difference does not apply to every individual.

The difference in means is descriptive; no test of differences between groups was performed.


In [ ]:
from scipy import stats

print('Normality of usage:')
print(stats.normaltest(ndf['usage'].dropna()))

print('Normality of savings:')
print(stats.normaltest(ndf['savings'].dropna()))

## Normality

For both variables, the p-value is below 0.05, so the null hypothesis of normality is rejected using the D'Agostino–Pearson test.


In [ ]:
for variable in ['usage', 'savings']:

    q1 = ndf[variable].quantile(0.25)
    q3 = ndf[variable].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = ndf[
        (ndf[variable] < lower) |
        (ndf[variable] > upper)
    ]

    print(variable)
    print('Lower bound:', lower)
    print('Upper bound:', upper)
    print('Potential outliers:', len(outliers))
    print()

## Outliers

The 1.5-times-interquartile-range rule identifies 11 potential outliers in usage and 9 in savings.

These records were retained because an outlier is not necessarily an error. Missing values were omitted only when the analysis required that variable.
